# ERA5 heat EDA — Madhya Pradesh & Kajiado

Exploratory analysis of monthly Tmax and heatwave-day counts (>=35 C for >=3 consecutive days) from `reanalysis-era5-single-levels` for two focus districts.

Two ways to run:

- **Offline fixture** (`use_fixture = True`): no network, runs instantly. Useful for previewing chart shapes.
- **Real ERA5**: set `use_fixture = False`. Requires `~/.cdsapirc` and accepted dataset licence on the CDS website. First run downloads the requested years; subsequent runs hit the on-disk cache and finish in seconds.

Sections below:
1. Load and shape
2. Summary statistics
3. Annual climatology (12-month cycle)
4. Distributions per calendar month
5. Year x month heatmaps
6. Long-term annual trends
7. Hottest months ranked
8. District comparison

In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys
import warnings

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pipelines" / "era5_heat" / "src").exists()
)
os.environ.setdefault(
    "MPLCONFIGDIR",
    str(project_root / "pipelines" / "era5_heat" / ".cache" / "matplotlib"),
)
sys.path.insert(0, str(project_root / "pipelines" / "era5_heat" / "src"))

warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from era5_heat import compute_heat_series, fixture_demo, PRESETS
from era5_heat.viz import monthly_heatmap, to_year_month_matrix

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

use_fixture = False
end_year = 2024
years = 5

def load(slug: str):
    if use_fixture:
        return fixture_demo(slug, years=years, end_year=end_year)
    p = PRESETS[slug]
    return compute_heat_series(district=p.name, bbox=p.bbox, years=years, end_year=end_year)

for slug, p in PRESETS.items():
    print(f'{slug:18s}  {p.name:18s}  {p.country:8s}  bbox(N,W,S,E)={p.bbox}')

## 1. Load and shape

We tack on `year`/`month_num` columns to make grouping cleaner.

In [ ]:
data = {}
for slug in PRESETS:
    df, meta = load(slug)
    df['month_ts'] = pd.to_datetime(df['month'])
    df['year'] = df['month_ts'].dt.year
    df['month_num'] = df['month_ts'].dt.month
    df['month_name'] = df['month_ts'].dt.strftime('%b')
    data[slug] = (df, meta)
    print(f"{slug:18s} rows={len(df):4d}  window={meta['window']['start_year']}-{meta['window']['end_year']}")

data['madhya-pradesh'][0].head()

## 2. Summary statistics

Side-by-side `describe()` for the three numeric columns.

- mean monthly Tmax should be plausible for the local climate
- absolute max should always be >= mean (we test this elsewhere too)
- heatwave days should be in `[0, 31]`

In [ ]:
summary = pd.concat(
    {slug: df[['tmax_monthly_max_c','tmax_monthly_mean_c','heatwave_days']].describe().round(2)
     for slug, (df, _) in data.items()},
    axis=1,
)
summary

## 3. Annual climatology

Each thin line is one calendar year's monthly Tmax cycle; the bold line is the 20-year median. Reveals seasonality (e.g. pre-monsoon peak for MP in Apr-May, double-peak around Feb and Sep-Oct near the equator for Kajiado).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (slug, (df, _)) in zip(axes, data.items()):
    for yr, sub in df.groupby('year'):
        ax.plot(sub['month_num'], sub['tmax_monthly_max_c'], color='#888', alpha=0.25, linewidth=0.9)
    median = df.groupby('month_num')['tmax_monthly_max_c'].median()
    ax.plot(median.index, median.values, color='#c0392b', linewidth=2.5, marker='o', label='20-yr median')
    ax.axhline(35.0, color='#2c3e50', linestyle='--', linewidth=1, alpha=0.6, label='35 C threshold')
    ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTH_LABELS)
    ax.set_title(f'{PRESETS[slug].name} ({PRESETS[slug].country})')
    ax.set_xlabel('Month'); ax.set_ylabel('Monthly absolute Tmax (C)')
    ax.legend(loc='upper right', fontsize=8)
fig.suptitle('Annual climatology — each light line is one year, bold line is 20-yr median', y=1.02)
fig.tight_layout()
fig

## 4. Distributions per calendar month

Boxplots show the spread of monthly Tmax across the 20 years for each calendar month. The wider/taller the box, the more interannual variability that month carries.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (slug, (df, _)) in zip(axes, data.items()):
    sns.boxplot(data=df, x='month_name', y='tmax_monthly_max_c', order=MONTH_LABELS,
                ax=ax, color='#d35400', linewidth=1, fliersize=2)
    ax.axhline(35.0, color='#2c3e50', linestyle='--', linewidth=1, alpha=0.6)
    ax.set_title(f'{PRESETS[slug].name} — interannual spread of monthly Tmax')
    ax.set_xlabel(''); ax.set_ylabel('Tmax (C)')
fig.tight_layout()
fig

## 5. Year x month heatmaps

The same data, but as a 2D grid: rows are years, columns are months, colour is intensity. Lets you spot anomaly years (a noticeably hotter or colder row) and structural shifts (a vertical band drifting upward over time).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (slug, (df, _)) in zip(axes, data.items()):
    mat = to_year_month_matrix(df, 'tmax_monthly_max_c')
    sns.heatmap(mat, ax=ax, cmap='inferno', annot=False, cbar_kws={'label':'C'},
                linewidths=0.3, linecolor='white')
    ax.set_title(f'{PRESETS[slug].name} — monthly absolute Tmax (C)')
    ax.set_xlabel(''); ax.set_ylabel('Year')
fig.tight_layout()
fig

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (slug, (df, _)) in zip(axes, data.items()):
    mat = to_year_month_matrix(df, 'heatwave_days')
    sns.heatmap(mat, ax=ax, cmap='magma', annot=True, fmt='d',
                annot_kws={'size': 7}, cbar_kws={'label':'days'},
                linewidths=0.3, linecolor='white')
    ax.set_title(f'{PRESETS[slug].name} — heatwave days per month (>=35 C, >=3-day runs)')
    ax.set_xlabel(''); ax.set_ylabel('Year')
fig.tight_layout()
fig

## 6. Long-term annual trends

Collapse to one number per year and look for trend. Left axis: annual peak Tmax (the hottest month of that year). Right axis: total heatwave days. A linear fit is overlaid for orientation only — 20 years is a short series for climate trend claims.

In [ ]:
annual = {}
for slug, (df, _) in data.items():
    a = df.groupby('year').agg(peak_tmax=('tmax_monthly_max_c','max'),
                                total_hw_days=('heatwave_days','sum'))
    annual[slug] = a

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (slug, a) in zip(axes, annual.items()):
    ax.plot(a.index, a['peak_tmax'], color='#c0392b', marker='o', label='annual peak Tmax (C)')
    # linear fit
    z = np.polyfit(a.index, a['peak_tmax'], 1)
    ax.plot(a.index, np.poly1d(z)(a.index), color='#c0392b', linestyle='--', alpha=0.5)
    ax.set_ylabel('peak Tmax (C)', color='#c0392b')
    ax.tick_params(axis='y', labelcolor='#c0392b')
    ax.set_title(f'{PRESETS[slug].name} (slope {z[0]:+.3f} C/yr)')

    ax2 = ax.twinx()
    ax2.bar(a.index, a['total_hw_days'], color='#34495e', alpha=0.25, label='total heatwave days')
    ax2.set_ylabel('heatwave days/yr', color='#34495e')
    ax2.tick_params(axis='y', labelcolor='#34495e')
    ax2.grid(False)
fig.tight_layout()
fig

## 7. Hottest months ranked

Top-10 hottest (year, month) pairs by absolute Tmax. Quick sanity check for plausibility — should land in the known hot season.

In [ ]:
from IPython.display import display, Markdown
for slug, (df, _) in data.items():
    top10 = (df.sort_values('tmax_monthly_max_c', ascending=False)
               .head(10)[['year','month_name','tmax_monthly_max_c','heatwave_days']]
               .reset_index(drop=True))
    display(Markdown(f"**{PRESETS[slug].name}** — top 10 hottest months"))
    display(top10)

## 8. District comparison

Same calendar, different climates — overlaid for direct visual comparison. The annual-cycle plot shows seasonal phase difference (MP peaks May; Kajiado is much flatter and peaks around Feb). The bar plot shows orders-of-magnitude difference in exposure to >=35 C days.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

palette = {'madhya-pradesh':'#c0392b', 'kajiado':'#2980b9'}
for slug, (df, _) in data.items():
    med = df.groupby('month_num')['tmax_monthly_max_c'].median()
    p10 = df.groupby('month_num')['tmax_monthly_max_c'].quantile(0.10)
    p90 = df.groupby('month_num')['tmax_monthly_max_c'].quantile(0.90)
    ax1.fill_between(med.index, p10, p90, color=palette[slug], alpha=0.15)
    ax1.plot(med.index, med.values, marker='o', color=palette[slug],
             label=f'{PRESETS[slug].name}')
ax1.axhline(35.0, color='#2c3e50', linestyle='--', linewidth=1, alpha=0.5)
ax1.set_xticks(range(1,13)); ax1.set_xticklabels(MONTH_LABELS)
ax1.set_title('Annual cycle — median with 10-90% interannual band')
ax1.set_ylabel('monthly Tmax (C)'); ax1.legend()

ann = pd.DataFrame({slug: a['total_hw_days'] for slug, a in annual.items()})
ann.plot(kind='bar', ax=ax2, color=[palette[c] for c in ann.columns], width=0.85)
ax2.set_title('Annual heatwave-day totals'); ax2.set_xlabel(''); ax2.set_ylabel('days/yr')
ax2.legend([PRESETS[c].name for c in ann.columns])
fig.tight_layout()
fig

---

Set `use_fixture = False` in the setup cell to fetch real ERA5 (`~/.cdsapirc` already configured). The first run will download the requested years per district and cache to `pipelines/era5_heat/.cache/`; re-runs are instant.